<a href="https://colab.research.google.com/github/viraj97-sl/ai-ml-ds-learning-hub/blob/master/02_ML_Engineer/beginner/06_first_ml_api.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# First End-to-End ML API — Train → Save → Serve

Build a complete ML system from scratch: train a model, save it with metadata, serve it via FastAPI, and containerize with Docker.

**Topics:** End-to-end pipeline, model artifact management, API integration, Docker deployment

In [ ]:
# Step 1: Train the model
import numpy as np
import pandas as pd
import joblib
import json
from pathlib import Path
from datetime import datetime
from sklearn.datasets import load_iris
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import classification_report, accuracy_score
import warnings; warnings.filterwarnings('ignore')

np.random.seed(42)

# Simple dataset: Iris classification
data = load_iris()
X, y = data.data, data.target
feature_names = list(data.feature_names)
class_names = list(data.target_names)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

# Train pipeline
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', GradientBoostingClassifier(n_estimators=50, max_depth=3, random_state=42)),
])

cv_scores = cross_val_score(pipeline, X_train, y_train, cv=5)
pipeline.fit(X_train, y_train)
test_accuracy = accuracy_score(y_test, pipeline.predict(X_test))

print('=== Iris Classifier Training ===')
print(f'CV accuracy: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}')
print(f'Test accuracy: {test_accuracy:.4f}')
print()
print(classification_report(y_test, pipeline.predict(X_test), target_names=class_names))

In [ ]:
# Step 2: Save model bundle
model_bundle = {
    'pipeline': pipeline,
    'metadata': {
        'model_name': 'iris-classifier',
        'version': '1.0.0',
        'training_date': datetime.utcnow().isoformat() + 'Z',
        'training_samples': len(X_train),
        'cv_accuracy': round(cv_scores.mean(), 4),
        'test_accuracy': round(test_accuracy, 4),
        'feature_names': feature_names,
        'class_names': class_names,
        'n_classes': len(class_names),
        'sklearn_version': __import__('sklearn').__version__,
    }
}

Path('models').mkdir(exist_ok=True)
model_path = 'models/iris_classifier_v1.pkl'
joblib.dump(model_bundle, model_path, compress=3)
print(f'Model saved to {model_path}')
print(f'Metadata: {json.dumps(model_bundle["metadata"], indent=2)}')

## 2. Complete FastAPI Application

In [ ]:
# Save this to src/iris_api.py and run:
# uvicorn src.iris_api:app --reload

iris_api_code = '''
# src/iris_api.py
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field
from typing import List, Dict
import joblib
import numpy as np
import time

app = FastAPI(
    title="Iris Species Classifier",
    version="1.0.0",
    description="Classify iris flowers by species using sepal/petal measurements",
)

# Global model state
model_bundle = None

@app.on_event("startup")
async def load_model():
    global model_bundle
    model_bundle = joblib.load("models/iris_classifier_v1.pkl")
    print(f"Loaded model: {model_bundle['metadata']['model_name']} v{model_bundle['metadata']['version']}")

# ===== Request / Response Models =====

class IrisFeatures(BaseModel):
    sepal_length_cm: float = Field(..., gt=0, le=20, example=5.1)
    sepal_width_cm:  float = Field(..., gt=0, le=20, example=3.5)
    petal_length_cm: float = Field(..., gt=0, le=20, example=1.4)
    petal_width_cm:  float = Field(..., gt=0, le=10, example=0.2)

class BatchRequest(BaseModel):
    flowers: List[IrisFeatures]

class PredictionResult(BaseModel):
    species: str
    confidence: float
    probabilities: Dict[str, float]
    latency_ms: float

# ===== Endpoints =====

@app.get("/")
def root():
    return {"service": "Iris Classifier", "docs": "/docs"}

@app.get("/health")
def health():
    if model_bundle is None:
        raise HTTPException(503, "Model not loaded")
    return {"status": "healthy", **model_bundle["metadata"]}

@app.post("/predict", response_model=PredictionResult)
def predict(flower: IrisFeatures):
    start = time.perf_counter()
    
    pipeline = model_bundle["pipeline"]
    class_names = model_bundle["metadata"]["class_names"]
    
    X = np.array([[flower.sepal_length_cm, flower.sepal_width_cm,
                   flower.petal_length_cm, flower.petal_width_cm]])
    
    probs = pipeline.predict_proba(X)[0]
    pred_idx = probs.argmax()
    
    return PredictionResult(
        species=class_names[pred_idx],
        confidence=round(float(probs[pred_idx]), 4),
        probabilities={cls: round(float(p), 4) for cls, p in zip(class_names, probs)},
        latency_ms=round((time.perf_counter() - start) * 1000, 2),
    )

@app.post("/predict/batch")
def predict_batch(request: BatchRequest):
    start = time.perf_counter()
    pipeline = model_bundle["pipeline"]
    class_names = model_bundle["metadata"]["class_names"]
    
    X = np.array([[f.sepal_length_cm, f.sepal_width_cm, f.petal_length_cm, f.petal_width_cm]
                  for f in request.flowers])
    
    probs_all = pipeline.predict_proba(X)
    results = []
    for probs in probs_all:
        pred_idx = probs.argmax()
        results.append({
            "species": class_names[pred_idx],
            "confidence": round(float(probs[pred_idx]), 4),
        })
    
    return {
        "predictions": results,
        "n_predictions": len(results),
        "latency_ms": round((time.perf_counter() - start) * 1000, 2),
    }
'''

# Write to file
Path('src').mkdir(exist_ok=True)
with open('src/iris_api.py', 'w') as f:
    f.write(iris_api_code)
print('src/iris_api.py written')
print()
print('Run API: uvicorn src.iris_api:app --reload --port 8001')
print()
print('Test with curl:')
print('''  curl -X POST http://localhost:8001/predict \\
    -H "Content-Type: application/json" \\
    -d \'{"sepal_length_cm": 5.1, "sepal_width_cm": 3.5, "petal_length_cm": 1.4, "petal_width_cm": 0.2}\'\n''')

## 3. Dockerfile for the API

In [ ]:
dockerfile_content = '''# Dockerfile
FROM python:3.11-slim

RUN useradd -m -u 1000 appuser

WORKDIR /app

# Install dependencies first (cached layer)
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

# Copy app
COPY src/ ./src/
COPY models/ ./models/

USER appuser
EXPOSE 8000
HEALTHCHECK --interval=30s CMD curl -f http://localhost:8000/health || exit 1
CMD ["uvicorn", "src.iris_api:app", "--host", "0.0.0.0", "--port", "8000"]
'''

requirements_content = '''fastapi==0.109.0
uvicorn[standard]==0.27.0
pydantic==2.5.3
scikit-learn==1.4.0
numpy==1.26.3
joblib==1.3.2
'''

with open('Dockerfile', 'w') as f:
    f.write(dockerfile_content)
with open('requirements.txt', 'w') as f:
    f.write(requirements_content)

print('Files created:')
print('  Dockerfile')
print('  requirements.txt')
print()
print('Build and run:')
print('  docker build -t iris-classifier:v1 .')
print('  docker run -p 8000:8000 iris-classifier:v1')
print()
print('Full directory structure:')
print('''
  iris-api/
  ├── Dockerfile
  ├── requirements.txt
  ├── .dockerignore
  ├── src/
  │   └── iris_api.py
  ├── models/
  │   └── iris_classifier_v1.pkl
  └── tests/
      └── test_api.py
''')